In [1]:
import os
os.chdir(r"C:\Users\shiya\practice\Protfolio\VibeCoding\creator-benchmarker")
print(os.getcwd())

C:\Users\shiya\practice\Protfolio\VibeCoding\creator-benchmarker


In [3]:
import json
from collections import Counter

with open("data/raw/channels_raw.json", encoding="utf-8") as f:
    data = json.load(f)

print(f"Total channels: {len(data)}\n")
counts = Counter(c["niche"] for c in data)
for niche, count in sorted(counts.items()):
    status = "✅" if count >= 7 else "⚠️"
    print(f"{status} {niche:<12} {count} channels")

Total channels: 73

✅ Beauty       10 channels
✅ Education    10 channels
✅ Finance      10 channels
✅ Fitness      7 channels
✅ Food         9 channels
✅ Gaming       8 channels
✅ Tech         10 channels
✅ Travel       9 channels


In [4]:
import pandas as pd

df = pd.DataFrame(data)
print(df[["title", "niche", "subscriber_count", "video_count", "view_count"]].head(10).to_string())

              title    niche  subscriber_count  video_count  view_count
0       Andrei Jikh  Finance           3050000         1128   399481200
1  Minority Mindset  Finance           2400000         2354   271359583
2    Graham Stephan  Finance           5150000         1470  1370157679
3        Meet Kevin  Finance           2050000         6124   761239728
4      Mark Tilbury  Finance           8000000          317  1960484743
5      Nate O'Brien  Finance           1280000          202    75135345
6     Ryan Scribner  Finance            866000          917    76704226
7   The Plain Bagel  Finance           1160000          267    94252012
8    Joseph Carlson  Finance            505000          451    42228375
9    Humphrey Talks  Finance                12            0           0


In [5]:
with open("data/raw/videos_raw.json", encoding="utf-8") as f:
    videos = json.load(f)

print(f"Total videos fetched: {len(videos)}")
print(f"Fields per video: {list(videos[0].keys())}")

Total videos fetched: 700
Fields per video: ['video_id', 'channel_id', 'niche', 'title', 'description', 'published_at', 'tags', 'duration', 'view_count', 'like_count', 'comment_count', 'engagement_rate']


In [6]:
video_counts = Counter(v["niche"] for v in videos)
for niche, count in sorted(video_counts.items()):
    print(f"  {niche:<12} {count} videos")

  Beauty       80 videos
  Education    100 videos
  Finance      90 videos
  Fitness      70 videos
  Food         90 videos
  Gaming       80 videos
  Tech         100 videos
  Travel       90 videos


In [7]:
vdf = pd.DataFrame(videos)
vdf[["title", "niche", "view_count", "like_count", "comment_count", "engagement_rate"]].head(10)

,title,niche,view_count,like_count,comment_count,engagement_rate
0,The $3 Trillion Market Where You Can’t Get You...,Finance,32692,917,29,2.8937
1,The U.S. Federal Reserve Is In Trouble,Finance,27360,755,29,2.8655
2,Oil Predicts Recessions,Finance,49001,1080,43,2.2918
3,Oil Prices Are Screaming “Recession”,Finance,59628,1278,65,2.2523
4,U.S. Could Reprice Gold,Finance,73331,2254,156,3.2865
5,Why The U.S. Economy Has Not Collapsed Yet,Finance,962512,25816,2477,2.9395
6,When AI Companies Say No to Governments,Finance,121909,2466,81,2.0893
7,How Oil Boosts the U.S. Dollar,Finance,214165,4670,162,2.2562
8,When Data Centers Become Military Targets,Finance,306136,3375,138,1.1475
9,Bitcoin’s Biggest Problem,Finance,106184,2631,95,2.5672


In [8]:
with open("data/raw/channels_classified.json", encoding="utf-8") as f:
    classified = json.load(f)

cdf = pd.DataFrame(classified)
print(f"Total classified: {len(cdf)}")
print(f"Overall NLP accuracy: {cdf['niche_match'].mean():.1%}")
print(f"\nNew fields added: {['nlp_niche', 'nlp_confidence', 'api_niche', 'niche_match']}")

Total classified: 73
Overall NLP accuracy: 86.3%

New fields added: ['nlp_niche', 'nlp_confidence', 'api_niche', 'niche_match']


In [9]:
mismatches = cdf[cdf["niche_match"] == False][["title", "api_niche", "nlp_niche", "nlp_confidence"]]
print(f"Mismatches: {len(mismatches)}")
print(mismatches.to_string())

Mismatches: 10
               title  api_niche  nlp_niche  nlp_confidence
4       Mark Tilbury    Finance  Education          0.1290
23        Wayne Goss     Beauty       Tech          0.1000
24             Hyram     Beauty  Education          0.1290
26    Tati Westbrook     Beauty    Unknown          0.0000
37          Fireship       Tech     Beauty          0.0714
44  Hardware Unboxed       Tech     Gaming          0.1071
55        Mark Wiens     Travel       Food          0.0333
56   Abroad in Japan     Travel     Gaming          0.0357
68   SmarterEveryDay  Education     Travel          0.0645
70            Vsauce  Education     Travel          0.0323


In [10]:
accuracy_by_niche = cdf.groupby("api_niche")["niche_match"].agg(["sum", "count", "mean"])
accuracy_by_niche.columns = ["matched", "total", "accuracy"]
accuracy_by_niche["accuracy"] = accuracy_by_niche["accuracy"].map("{:.0%}".format)
print(accuracy_by_niche.to_string())

           matched  total accuracy
api_niche                         
Beauty           7     10      70%
Education        8     10      80%
Finance          9     10      90%
Fitness          7      7     100%
Food             9      9     100%
Gaming           8      8     100%
Tech             8     10      80%
Travel           7      9      78%


In [11]:
mismatches = cdf[cdf["niche_match"] == False][["title", "api_niche", "nlp_niche", "nlp_confidence"]]
print(f"Total mismatches: {len(mismatches)}\n")
print(mismatches.to_string())

Total mismatches: 10

               title  api_niche  nlp_niche  nlp_confidence
4       Mark Tilbury    Finance  Education          0.1290
23        Wayne Goss     Beauty       Tech          0.1000
24             Hyram     Beauty  Education          0.1290
26    Tati Westbrook     Beauty    Unknown          0.0000
37          Fireship       Tech     Beauty          0.0714
44  Hardware Unboxed       Tech     Gaming          0.1071
55        Mark Wiens     Travel       Food          0.0333
56   Abroad in Japan     Travel     Gaming          0.0357
68   SmarterEveryDay  Education     Travel          0.0645
70            Vsauce  Education     Travel          0.0323


In [12]:
from sqlalchemy import create_engine, text

engine = create_engine("sqlite:///data/db/creator_benchmarker.db")

with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT
            niche,
            total_channels,
            ROUND(avg_subscribers / 1000000, 2)  AS avg_subs_millions,
            ROUND(avg_engagement_rate, 2)         AS avg_eng_rate,
            ROUND(avg_videos_per_month, 1)        AS uploads_per_month
        FROM niche_benchmarks
        ORDER BY avg_subscribers DESC
    """))
    df = pd.DataFrame(result.fetchall(), columns=result.keys())
    print(df.to_string(index=False))

    niche  total_channels  avg_subs_millions  avg_eng_rate  uploads_per_month
   Gaming               8              34.93          4.00                7.1
Education              10              21.19          4.61                5.0
     Tech              10              10.38          4.02                8.2
  Fitness               7               6.36          3.61                5.8
     Food               9               4.98          4.23                6.9
   Beauty              10               3.36          7.11                4.6
   Travel               9               3.00          4.86                6.3
  Finance              10               2.45          3.14                6.1


In [ ]:
from src.analytics import get_engine, get_top_videos_per_niche

engine = get_engine()
df = get_top_videos_per_niche(engine, limit=5)

print(f"Total rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"Niches present: {df['niche'].unique().tolist()}")
print(df.head(10).to_string())

In [14]:
from src.analytics import get_engine, get_top_videos_per_niche

engine = get_engine()
df = get_top_videos_per_niche(engine, limit=5)

print(f"Total rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(df.head(3).to_string())

Total rows: 40
Columns: ['title', 'channel', 'view_count', 'like_count', 'comment_count', 'engagement_rate', 'published_at']
                                                     title          channel  view_count  like_count  comment_count  engagement_rate          published_at
0             trying HOLOGRAPHIC skin.. cause why not? 🤔💿✨  NikkieTutorials    13051352      751215           4054             5.79  2026-01-04T18:12:03Z
1  this filter decides the ORDER of my MAKEUP ROUTINE! 😵‍💫  NikkieTutorials     3493595      245252           3390             7.12  2026-01-20T18:15:39Z
2                      temporary GLITTER eye tattoos?! 🫢👀✨  NikkieTutorials     2560281      152577           1271             6.01  2025-12-21T18:33:13Z


In [16]:
import importlib
import src.analytics
importlib.reload(src.analytics)

from src.analytics import get_engine, get_top_videos_per_niche

engine = get_engine()
df = get_top_videos_per_niche(engine, limit=5)

print(f"Total rows     : {len(df)}")
print(f"Columns        : {list(df.columns)}")
print(f"Niche present  : {'niche' in df.columns}")
print(f"Niches          : {sorted(df['niche'].unique().tolist())}")
print(f"\nRows per niche:")
print(df.groupby('niche').size().to_string())

Debug — columns after SQL: ['niche', 'title', 'channel', 'view_count', 'like_count', 'comment_count', 'engagement_rate', 'published_at']
Debug — row count: 700
Total rows     : 40
Columns        : ['niche', 'title', 'channel', 'view_count', 'like_count', 'comment_count', 'engagement_rate', 'published_at']
Niche present  : True
Niches          : ['Beauty', 'Education', 'Finance', 'Fitness', 'Food', 'Gaming', 'Tech', 'Travel']

Rows per niche:
niche
Beauty       5
Education    5
Finance      5
Fitness      5
Food         5
Gaming       5
Tech         5
Travel       5


In [2]:
import os
print(os.path.exists(".\\data\\raw\\channels_raw.json"))


True


In [3]:

print(os.getcwd())

c:\Users\shiya\practice\Protfolio\VibeCoding\creator-benchmarker


In [4]:
print(os.path.exists("data/raw/channels_raw.json"))

True


In [2]:
print("kernel working")

kernel working


In [ ]:
import json

with open("data/raw/channels_raw.json", encoding="utf-8") as f:
    data = json.load(f)

print(f"Total channels fetched: {len(data)}")
print(f"First channel  : {data[0]['title']}")
print(f"Niche          : {data[0]['niche']}")
print(f"Subscribers    : {data[0]['subscriber_count']:,}")

In [8]:
import json
import os
from dotenv import load_dotenv
from googleapiclient.discovery import build

load_dotenv()
API_KEY = os.getenv("YOUTUBE_API_KEY")
youtube = build("youtube", "v3", developerKey=API_KEY)

# Full seed list from fetch_channels.py
CHANNELS_BY_NICHE = {
    "Finance": [
        "UCVHFbw7woebKtFFixitbcWQ",
        "UCL8w_A8t3eqkJ_FCbMEgizg",
        "UCV6KDgJskWaEckne5aPA0aQ",
        "UCnMn36GT_H0X-w5_ckLtlgQ",
        "UCEZJ_9FRRfyBLFcJNzBYqyA",
        "UCTLrDnRBgge9bMEbMRVWJQA",
        "UC3mjMoJuFnjYRBLon_6njbQ",
        "UCbmNph6atAoGfqLoCL_duAg",
        "UCFCEuCsyWP0YkP3CZ3Mr01Q",
        "UCEAZeUIeJs9HFMEcgGDSjbA",
    ],
    "Fitness": [
        "UCERm5yFZ1SptUcqRRR5bwa",
        "UCe0TLA0EsQbE-MjuHXevPRg",
        "UC68TLK0mAEzUyHu6FqnvWVA",
        "UCpQ34nge0-OeEnNFnNjUYhg",
        "UCfQgsKhHjSyRLOp9mnffqVg",
        "UCIJiTR0TFd-eEEjQVCuDnDQ",
        "UCaBqRxHEMomgFU-AkSfodCqA",
        "UC-axK-PT_b0JKioGbqdDK5A",
        "UCDEHbE4YUTCNPqJ3ckZqjSg",
        "UCDKbEl4AzZzQmkISVpq1ItA",
    ],
    "Beauty": [
        "UCgFv_WL0p_OrGMCj8DPpJQw",
        "UCYbyGGFGSqH2pOxC-QWRaBA",
        "UCnEw2Qm9-SHFxLALHBaFHtg",
        "UCHtDMFBmq_iXClFRNinj_Cg",
        "UCddiUEpeqJcYeBxX1IVBKvQ",
        "UCE_M8A5yxnLfW0KghEeajjw",
        "UC3yBbm1pFGPyWNNKwGJeHdw",
        "UCgFv_WL0p_OrGMCj8DPpJQw",
        "UCiEEF51uRAeZeCo8CJFhGWw",
        "UC_r2H84D_E_0BtSCGzOHiJw",
    ],
    "Gaming": [
        "UCX6OQ3DkcsbYNE6H8uQQuVA",
        "UCYzPXprvl5Y-Sf0g4vX-m6g",
        "UCo8bcnLyZH8tBIH9V1mLgqQ",
        "UCwRqWnW5ZkVaP_lZF7caZ-g",
        "UC2C_jShtL725hvbm1arSV9w",
        "UCJHA_jMfCvEnv-3kRjTCQXw",
        "UCRijo3ddMTht_IHyNSNXpNQ",
        "UCq-Fj5jknLsUf-MWSy4_brA",
        "UCVtymbOY_bnNZiHMDNMDdaQ",
        "UCam8T03EOFBsNdR0thrFHdQ",
    ],
    "Tech": [
        "UCBcRF18a7Qf58cCRy5xuWwQ",
        "UCXuqSBlHAE6Xw-yeJA0Tunw",
        "UCHnyfMqiRRG1u-2MsSQLbXA",
        "UC9-y-6csu5WGm29I7JiwpnA",
        "UCo8bcnLyZH8tBIH9V1mLgqQ",
        "UCddiUEpeqJcYeBxX1IVBKvQ",
        "UCTqiPyDB5DkC_dsnpOuIpOA",
        "UC3yEBaQDaA7saFCLjFOoGkQ",
        "UCe_CCR7HZqlf_n2GxDn0_Cg",
        "UCoxcjq-8xIDTYp3uz647V5A",
    ],
    "Food": [
        "UCJHA_jMfCvEnv-3kRjTCQXw",
        "UCJFp8uSYCjXOMnkUyb3CQ3Q",
        "UCRIZtPl9nb9RiXc9btSTQNw",
        "UC8Q7XEy86Q7T-3kNpNnYUFg",
        "UCWD4uIzOtDOlOOsDDmm98uQ",
        "UCmmPgObSUPw1HL2lq6H4ffQ",
        "UC2bkHVIDjXS7sgrgjFtzOXQ",
        "UCVYamHliCI9rw1tHR1xbkfw",
        "UCdWKgnFh0NRKQ88jFBygV-Q",
        "UCJFp8uSYCjXOMnkUyb3CQ3Q",
    ],
    "Travel": [
        "UCt_WhTiBukJoiFoYX0Gp8WA",
        "UCnTsUMsFgnglFamo_kYFhCQ",
        "UCX_xJGJBtbMFPvlAk1DBU4Q",
        "UCiDJtJKMICpb9B1qf7qjEOA",
        "UCqUa8gr0plRaWnPsEAnKqDg",
        "UCVYamHliCI9rw1tHR1xbkfw",
        "UC9RM-iSvTu1uPJb8X5yp3EQ",
        "UCPUuHHUAGJtMbQZhMSNOvXA",
        "UCzWQYUVCpZqtN93H8RR44Qw",
        "UCRijo3ddMTht_IHyNSNXpNQ",
    ],
    "Education": [
        "UCsooa4yRKGN_zEE8iknghZA",
        "UCWX3yGbODI3RMbCOjEGpTXA",
        "UCHnyfMqiRRG1u-2MsSQLbXA",
        "UCtESv1e7ntJaLJYKIO1FoYw",
        "UC7YOGHUfC1Tb2GcfshLUyWA",
        "UCX6OQ3DkcsbYNE6H8uQQuVA",
        "UCZYTClx2T1of7BRZ86-8fow",
        "UC6107grRI4m0o2-emgoDnAA",
        "UCG-KntY7aVnIGXYEBQvmBAQ",
        "UCoxcjq-8xIDTYp3uz647V5A",
    ],
}

# Check each ID and report which ones return no data
bad_ids = []
for niche, ids in CHANNELS_BY_NICHE.items():
    for channel_id in ids:
        response = youtube.channels().list(
            part="snippet",
            id=channel_id
        ).execute()
        if not response.get("items"):
            bad_ids.append({"niche": niche, "channel_id": channel_id})
            print(f"❌ BAD  | {niche:<12} | {channel_id}")
        else:
            title = response["items"][0]["snippet"]["title"]
            print(f"✅ OK   | {niche:<12} | {title}")

print(f"\nTotal bad IDs: {len(bad_ids)}")


❌ BAD  | Finance      | UCVHFbw7woebKtFFixitbcWQ
❌ BAD  | Finance      | UCL8w_A8t3eqkJ_FCbMEgizg
✅ OK   | Finance      | Graham Stephan
✅ OK   | Finance      | Financial Education
❌ BAD  | Finance      | UCEZJ_9FRRfyBLFcJNzBYqyA
❌ BAD  | Finance      | UCTLrDnRBgge9bMEbMRVWJQA
✅ OK   | Finance      | Ryan Scribner
✅ OK   | Finance      | Talks at Google
✅ OK   | Finance      | The Plain Bagel
❌ BAD  | Finance      | UCEAZeUIeJs9HFMEcgGDSjbA
❌ BAD  | Fitness      | UCERm5yFZ1SptUcqRRR5bwa
❌ BAD  | Fitness      | UCe0TLA0EsQbE-MjuHXevPRg
❌ BAD  | Fitness      | UC68TLK0mAEzUyHu6FqnvWVA
❌ BAD  | Fitness      | UCpQ34nge0-OeEnNFnNjUYhg
✅ OK   | Fitness      | Renaissance Periodization
❌ BAD  | Fitness      | UCIJiTR0TFd-eEEjQVCuDnDQ
❌ BAD  | Fitness      | UCaBqRxHEMomgFU-AkSfodCqA
❌ BAD  | Fitness      | UC-axK-PT_b0JKioGbqdDK5A
❌ BAD  | Fitness      | UCDEHbE4YUTCNPqJ3ckZqjSg
❌ BAD  | Fitness      | UCDKbEl4AzZzQmkISVpq1ItA
❌ BAD  | Beauty       | UCgFv_WL0p_OrGMCj8DPpJQw
❌ BAD  | Beaut

In [9]:
import time

def get_channel_id(youtube, channel_name: str) -> dict | None:
    """
    Search YouTube for a channel by name and return its ID and title.
    """
    try:
        response = youtube.search().list(
            part="snippet",
            q=channel_name,
            type="channel",
            maxResults=1
        ).execute()

        items = response.get("items", [])
        if not items:
            print(f"❌ Not found: {channel_name}")
            return None

        channel_id = items[0]["snippet"]["channelId"]
        title = items[0]["snippet"]["title"]
        print(f"✅ Found: {title:<35} | ID: {channel_id}")
        return {"name": channel_name, "channel_id": channel_id, "title": title}

    except Exception as e:
        print(f"❌ Error searching {channel_name}: {e}")
        return None


# Define replacement channels by niche — easy to add or remove names
REPLACEMENT_CHANNELS = {
    "Finance": [
        "Investopedia",
        "Humphrey Yang",
        "Andrei Jikh",
        "Graham Stephan",
        "Mark Tilbury",
        "Nate O Brien Finance",
        "Ryan Scribner",
        "The Plain Bagel",
        "Joseph Carlson",
        "Minority Mindset",
    ],
    "Fitness": [
        "Jeff Nippard",
        "Athlean X",
        "Jeremy Ethier",
        "Hybrid Calisthenics",
        "Austin Dunham",
        "Calisthenicmovement",
        "Chris Heria",
        "Magnus Midtbo",
        "Renaissance Periodization",
        "Alan Thrall",
    ],
    "Beauty": [
        "NikkieTutorials",
        "Safiya Nygaard",
        "Jackie Aina",
        "Stephanie Lange",
        "Hindash",
        "Lisa Eldridge",
        "Wayne Goss",
        "Hyram Skincare",
        "Samantha Ravndahl",
        "Tati Westbrook",
    ],
    "Gaming": [
        "Markiplier",
        "Jacksepticeye",
        "Ninja",
        "Pokimane",
        "Valkyrae",
        "Dude Perfect",
        "PewDiePie",
        "Dream",
        "Technoblade",
        "Ludwig",
    ],
    "Tech": [
        "Marques Brownlee",
        "Linus Tech Tips",
        "Fireship",
        "NetworkChuck",
        "Unbox Therapy",
        "Mrwhosetheboss",
        "Dave2D",
        "Computerphile",
        "TechLinked",
        "Hardware Unboxed",
    ],
    "Food": [
        "Binging with Babish",
        "Joshua Weissman",
        "Ethan Chlebowski",
        "Internet Shaquille",
        "Pro Home Cooks",
        "Guga Foods",
        "Tasting History",
        "Food Wishes",
        "Mythical Kitchen",
        "Brian Lagerstrom",
    ],
    "Travel": [
        "Kara and Nate",
        "Lost LeBlancs",
        "Mark Wiens",
        "Abroad in Japan",
        "Sailing La Vagabonde",
        "Fearless and Far",
        "Wolters World",
        "Samuel and Audrey",
        "Hopscotch the Globe",
        "Chris Abroad",
    ],
    "Education": [
        "TED Ed",
        "Kurzgesagt",
        "Veritasium",
        "Crash Course",
        "3Blue1Brown",
        "SmarterEveryDay",
        "Mark Rober",
        "Vsauce",
        "MinutePhysics",
        "Scott Manley",
    ],
}

# Search YouTube for each channel name and collect results
results_by_niche = {}
total = sum(len(names) for names in REPLACEMENT_CHANNELS.values())
count = 0

for niche, names in REPLACEMENT_CHANNELS.items():
    print(f"\n── {niche} ──────────────────────────────")
    results_by_niche[niche] = []
    for name in names:
        count += 1
        result = get_channel_id(youtube, name)
        if result:
            results_by_niche[niche].append(result["channel_id"])
        time.sleep(0.5)  # Respect API rate limits

print(f"\n\nDone! Found IDs for channels across {len(results_by_niche)} niches")


── Finance ──────────────────────────────
✅ Found: Investopedia                        | ID: UCvwFhI0mrIWDiZUabRapS5Q
✅ Found: Humphrey Yang                       | ID: UCFBpVaKCC0ajGps1vf0AgBg
✅ Found: Andrei Jikh                         | ID: UCGy7SkBjcIAgTiwkXEtPnYg
✅ Found: Graham Stephan                      | ID: UCV6KDgJskWaEckne5aPA0aQ
✅ Found: Mark Tilbury                        | ID: UCxgAuX3XZROujMmGphN_scA
✅ Found: Nate O'Brien                        | ID: UCO3tlaeZ6Z0ZN5frMZI3-uQ
✅ Found: Ryan Scribner                       | ID: UC3mjMoJuFnjYRBLon_6njbQ
✅ Found: The Plain Bagel                     | ID: UCFCEuCsyWP0YkP3CZ3Mr01Q
✅ Found: Joseph Carlson                      | ID: UCbta0n8i6Rljh0obO7HzG9A
✅ Found: Minority Mindset                    | ID: UCT3EznhW_CNFcfOlyDNTLLw

── Fitness ──────────────────────────────
✅ Found: Jeff Nippard                        | ID: UC68TLK0mAEzUyHx5x5k-S1Q
✅ Found: ATHLEAN-X™                          | ID: UCe0TLA0EsQbE-MjuHXevj2A
✅ 

In [10]:
print("CHANNELS_BY_NICHE = {")
for niche, ids in results_by_niche.items():
    print(f'    "{niche}": [')
    for channel_id in ids:
        print(f'        "{channel_id}",')
    print("    ],")
print("}")

CHANNELS_BY_NICHE = {
    "Finance": [
        "UCvwFhI0mrIWDiZUabRapS5Q",
        "UCFBpVaKCC0ajGps1vf0AgBg",
        "UCGy7SkBjcIAgTiwkXEtPnYg",
        "UCV6KDgJskWaEckne5aPA0aQ",
        "UCxgAuX3XZROujMmGphN_scA",
        "UCO3tlaeZ6Z0ZN5frMZI3-uQ",
        "UC3mjMoJuFnjYRBLon_6njbQ",
        "UCFCEuCsyWP0YkP3CZ3Mr01Q",
        "UCbta0n8i6Rljh0obO7HzG9A",
        "UCT3EznhW_CNFcfOlyDNTLLw",
    ],
    "Fitness": [
        "UC68TLK0mAEzUyHx5x5k-S1Q",
        "UCe0TLA0EsQbE-MjuHXevj2A",
        "UCERm5yFZ1SptUEU4wZ2vJvw",
        "UCeJFgNahi--FKs0oJyeRDEw",
        "UCRG_IQW6Yw5JmtTlKz8PBcg",
        "UCZIIRX8rkNjVpP-oLMHpeDw",
        "UCaBqRxHEMomgFU-AkSfodCw",
        "UC_gSotrFVZ_PiAxo3fTQVuQ",
        "UCfQgsKhHjSyRLOp9mnffqVg",
        "UCRLOLGZl3-QTaJfLmAKgoAw",
    ],
    "Beauty": [
        "UCzTKskwIc_-a0cGvCXA848Q",
        "UCbAwSkqJ1W_Eg7wr3cp5BUA",
        "UCzJIliq68IHSn-Kwgjeg2AQ",
        "UCZQ0XPE7wxMafUf8dIHOxgQ",
        "UCs7hbbgJFer2AcKtwAV5Bwg",
        "UCFgh